In [ ]:
# Install required libraries
!pip install openai gradio requests pandas matplotlib -q

In [ ]:
import openai
import requests
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata

# Load API keys
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
TMDB_API_KEY   = userdata.get('TMDB_API_KEY')
OMDB_API_KEY   = userdata.get('OMDB_API_KEY')

# Set up OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)
print("✓ OpenAI client ready")

# Test TMDB
# tmdb_test = requests.get(
#     f"https://api.themoviedb.org/3/movie/550?api_key={TMDB_API_KEY}"
# )
# print(f"✓ TMDB API: {tmdb_test.json()['title']}")

# # Test OMDb
# omdb_test = requests.get(
#     f"http://www.omdbapi.com/?t=Inception&apikey={OMDB_API_KEY}"
# )
# print(f"✓ OMDb API: {omdb_test.json()['Title']}")


✓ OpenAI client ready
✓ TMDB API: Fight Club
✓ OMDb API: Inception

🎬 All APIs ready — CineMatch setup complete!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time

films = []

print("Fetching hidden gem films from TMDB...")

for page in range(1, 16):  # 15 pages x 20 films = ~300 films
    url = "https://api.themoviedb.org/3/discover/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "vote_average.gte": 7.0,
        "vote_count.gte": 1000,
        "vote_count.lte": 10000,
        "sort_by": "vote_average.desc",
        "page": page
    }
    response = requests.get(url, params=params)
    data = response.json()

    for film in data.get('results', []):
        films.append({
            "title": film.get("title"),
            "year": film.get("release_date", "")[:4],
            "genre": str(film.get("genre_ids", [])),
            "imdb_rating": film.get("vote_average"),
            "num_votes": film.get("vote_count"),
            "overview": film.get("overview", "")[:150],
            "hidden_gem_score": round(
                ((film.get("vote_average", 0) / 10) * 0.6 +
                (1 - film.get("vote_count", 10000) / 10000) * 0.4) * 10,
                1
            )
        })

    time.sleep(0.3)
    print(f"  Page {page}/15 — {len(films)} films collected")

# Save to dataframe
df_films = pd.DataFrame(films)

# Remove duplicates
df_films = df_films.drop_duplicates(subset='title')

print(f"\n✓ Dataset complete: {len(df_films)} unique films")
print(f"✓ Rating range: {df_films['imdb_rating'].min()} "
      f"— {df_films['imdb_rating'].max()}")
print(f"✓ Year range: {df_films['year'].min()} "
      f"— {df_films['year'].max()}")

# Save to Google Drive
output_path = '/content/drive/MyDrive/CineMatch/cinematch_dataset.csv'
df_films.to_csv(output_path, index=False)
print(f"\n✓ Saved to Google Drive")
print(df_films[['title', 'year', 'imdb_rating',
                 'num_votes', 'hidden_gem_score']].head(10))


# df_films = pd.read_csv(
#     '/content/drive/MyDrive/CineMatch/cinematch_dataset.csv'
# )
# print(f"✓ Dataset loaded: {len(df_films)} films")

✓ Dataset loaded: 223 films


In [ ]:
# TMDB genre ID to name mapping
GENRE_MAP = {
    28: "Action", 12: "Adventure", 16: "Animation",
    35: "Comedy", 80: "Crime", 99: "Documentary",
    18: "Drama", 10751: "Family", 14: "Fantasy",
    36: "History", 27: "Horror", 10402: "Music",
    9648: "Mystery", 10749: "Romance", 878: "Sci-Fi",
    10770: "TV Movie", 53: "Thriller", 10752: "War",
    37: "Western"
}

def convert_genres(genre_str):
    try:
        ids = eval(genre_str)
        names = [GENRE_MAP.get(i, str(i)) for i in ids]
        return ', '.join(names)
    except:
        return genre_str

# Fix genres
df_films = pd.read_csv(
    '/content/drive/MyDrive/CineMatch/cinematch_dataset.csv'
)
df_films['genre'] = df_films['genre'].apply(convert_genres)

# Remove pre-1970 films
df_films = df_films[df_films['year'].astype(str) >= '1970']

# Remove post-2024 films
df_films = df_films[df_films['year'].astype(str) <= '2024']

# Remove well-known classics GPT already knows well
# (films appearing in every "greatest films" list)
known_classics = [
    '12 Angry Men', 'Seven Samurai',
    'The Good, the Bad and the Ugly',
    'Grave of the Fireflies'
]
df_films = df_films[~df_films['title'].isin(known_classics)]

# Add director column placeholder
df_films['director'] = 'TBC via TMDB'

# Reorder columns cleanly
df_films = df_films[[
    'title', 'year', 'director', 'genre',
    'imdb_rating', 'num_votes', 'hidden_gem_score', 'overview'
]]

# Save clean version
output_path = (
    '/content/drive/MyDrive/CineMatch/cinematch_dataset.csv'
)
df_films.to_csv(output_path, index=False)

print(f"✓ Clean dataset: {len(df_films)} films")
print(f"✓ Year range: {df_films['year'].min()}"
      f" — {df_films['year'].max()}")
print(f"✓ Genres fixed — sample:")
print(df_films['genre'].head(5).tolist())
print(f"\nSample films:")
print(df_films[['title', 'year', 'genre',
                 'imdb_rating', 'num_votes']].head(10).to_string())

✓ Clean dataset: 223 films
✓ Year range: 1973 — 2024
✓ Genres fixed — sample:
['Comedy, Drama, Romance', 'Drama, Crime', 'Drama, Romance', 'Comedy, Drama, Fantasy, Romance', 'Romance, Drama']

Sample films:
                                 title  year                                 genre  imdb_rating  num_votes
0          Dilwale Dulhania Le Jayenge  1995                Comedy, Drama, Romance        8.500       4575
1                          City of God  2002                          Drama, Crime        8.429       8119
2                      Cinema Paradiso  1988                        Drama, Romance        8.426       4799
3                         A Dog's Will  2000       Comedy, Drama, Fantasy, Romance        8.412       1214
4                    Gabriel's Inferno  2020                        Romance, Drama        8.400       2453
5            A Silent Voice: The Movie  2016             Animation, Drama, Romance        8.400       4446
6          Once Upon a Time in America  1984

In [ ]:
import json
import time

# Convert dataset to text chunks for RAG
# Each film becomes a readable text passage
# This is what gets stored in the vector store

def film_to_text(row):
    return (
        f"Title: {row['title']}\n"
        f"Year: {row['year']}\n"
        f"Genre: {row['genre']}\n"
        f"IMDB Rating: {row['imdb_rating']}/10\n"
        f"Number of Votes: {row['num_votes']} "
        f"(low = more obscure)\n"
        f"Hidden Gem Score: {row['hidden_gem_score']}/10\n"
        f"Overview: {row['overview']}\n"
    )

# Convert every film to a text passage
df_films['text'] = df_films.apply(film_to_text, axis=1)

# Save as a single text file — one film per block
rag_text_path = (
    '/content/drive/MyDrive/CineMatch/cinematch_rag.txt'
)

with open(rag_text_path, 'w', encoding='utf-8') as f:
    for _, row in df_films.iterrows():
        f.write(row['text'])
        f.write("\n---\n")  # separator between films

print(f"✓ RAG text file created")
print(f"✓ {len(df_films)} film entries written")
print(f"\nSample entry:")
print(df_films.iloc[0]['text'])

✓ RAG text file created
✓ 223 film entries written

Sample entry:
Title: Dilwale Dulhania Le Jayenge
Year: 1995
Genre: Comedy, Drama, Romance
IMDB Rating: 8.5/10
Number of Votes: 4575 (low = more obscure)
Hidden Gem Score: 7.3/10
Overview: Raj is a rich, carefree, happy-go-lucky second generation NRI. Simran is the daughter of Chaudhary Baldev Singh, who in spite of being an NRI is very 



In [ ]:
# Upload the file to OpenAI

print("Uploading RAG file to OpenAI...")

with open(rag_text_path, 'rb') as f:
    rag_file = client.files.create(
        file=f,
        purpose='assistants'
    )

print(f"✓ File uploaded successfully")
print(f"✓ File ID: {rag_file.id}")


RAG_FILE_ID = rag_file.id
print(f"\nRAG_FILE_ID = '{RAG_FILE_ID}'")


# Saved IDs — no need to re-upload or recreate
# RAG_FILE_ID = "file-UZ1XPhbGiNPGX1JU9hbp5A"
# VECTOR_STORE_ID = "vs_69f0d91457ac819185f331656b10b225"

# print(f"✓ File ID: {RAG_FILE_ID}")
# print(f"✓ Vector Store ID: {VECTOR_STORE_ID}")

✓ File ID: file-UZ1XPhbGiNPGX1JU9hbp5A
✓ Vector Store ID: vs_69f0d91457ac819185f331656b10b225


In [ ]:
# Create the vector store

print("Creating vector store...")

vector_store = client.vector_stores.create(
    name="CineMatch Film Database"
)

print(f"✓ Vector store created")
print(f"✓ Vector Store ID: {vector_store.id}")

# Attach the file to the vector store
client.vector_stores.file_batches.create(
    vector_store_id=vector_store.id,
    file_ids=[RAG_FILE_ID]
)

print(f"✓ File attached to vector store")
print(f"✓ Indexing started...")

# Wait for indexing to complete
print("\nWaiting for indexing...")
while True:
    status = client.vector_stores.retrieve(vector_store.id)
    print(f"  Status: {status.status}")
    if status.status == "completed":
        break
    time.sleep(3)

VECTOR_STORE_ID = vector_store.id
print(f"\n✓ Vector store ready!")
print(f"✓ VECTOR_STORE_ID = '{VECTOR_STORE_ID}'")

In [ ]:
# ============================================
# FUNCTION 1 — Get live movie details (TMDB)
# ============================================
def get_movie_details(title: str) -> dict:
    """Fetch live movie details from TMDB"""
    url = "https://api.themoviedb.org/3/search/movie"
    params = {"api_key": TMDB_API_KEY, "query": title}
    response = requests.get(url, params=params)
    results = response.json().get('results', [])

    if not results:
        return {"error": f"No results found for '{title}'"}

    movie = results[0]
    movie_id = movie['id']

    # Get full details including runtime
    detail_url = (
        f"https://api.themoviedb.org/3/movie/{movie_id}"
    )
    detail_params = {
        "api_key": TMDB_API_KEY,
        "append_to_response": "credits"
    }
    detail = requests.get(
        detail_url, params=detail_params
    ).json()

    # Extract director from credits
    director = "Unknown"
    for crew in detail.get(
        'credits', {}
    ).get('crew', []):
        if crew['job'] == 'Director':
            director = crew['name']
            break

    # Extract top 3 cast
    cast = [
        c['name'] for c in
        detail.get('credits', {}).get('cast', [])[:3]
    ]

    return {
        "title": detail.get('title'),
        "year": detail.get('release_date', '')[:4],
        "director": director,
        "cast": ', '.join(cast),
        "runtime": f"{detail.get('runtime', 'N/A')} mins",
        "rating": detail.get('vote_average'),
        "votes": detail.get('vote_count'),
        "overview": detail.get('overview', '')[:200],
        "genres": ', '.join(
            [g['name'] for g in detail.get('genres', [])]
        )
    }


# ============================================
# FUNCTION 2 — Get awards data (OMDb)
# ============================================
def get_awards_data(title: str) -> dict:
    """Fetch awards and additional info from OMDb"""
    url = "http://www.omdbapi.com/"
    params = {"t": title, "apikey": OMDB_API_KEY}
    response = requests.get(url, params=params)
    data = response.json()

    if data.get('Response') == 'False':
        return {"error": f"No OMDb data found for '{title}'"}

    return {
        "title": data.get('Title'),
        "awards": data.get('Awards', 'No awards data'),
        "imdb_rating": data.get('imdbRating'),
        "metacritic": data.get('Metascore'),
        "rated": data.get('Rated'),
        "country": data.get('Country'),
        "language": data.get('Language'),
        "box_office": data.get('BoxOffice', 'N/A')
    }


# ============================================
# FUNCTION 3 — Weather based mood (Open-Meteo)
# ============================================
def get_weather_mood(city: str) -> dict:
    """Get current weather to suggest mood-matched films"""

    # City coordinates lookup (top cities)
    city_coords = {
        "dublin": (53.3498, -6.2603),
        "london": (51.5074, -0.1278),
        "new york": (40.7128, -74.0060),
        "paris": (48.8566, 2.3522),
        "tokyo": (35.6762, 139.6503),
        "sydney": (-33.8688, 151.2093),
        "toronto": (43.6532, -79.3832),
        "berlin": (52.5200, 13.4050),
        "mumbai": (19.0760, 72.8777),
        "cork": (51.8985, -8.4756),
        "galway": (53.2707, -9.0568),
    }

    city_lower = city.lower().strip()
    coords = city_coords.get(
        city_lower, (53.3498, -6.2603)  # default Dublin
    )

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coords[0],
        "longitude": coords[1],
        "current": "temperature_2m,weathercode",
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    data = response.json()

    temp = data['current']['temperature_2m']
    code = data['current']['weathercode']

    # Weather code to mood mapping
    if code == 0:
        condition = "clear and sunny"
        suggested_mood = "uplifting or romantic"
    elif code in [1, 2, 3]:
        condition = "partly cloudy"
        suggested_mood = "thought-provoking or humorous"
    elif code in [45, 48]:
        condition = "foggy"
        suggested_mood = "dark or melancholic"
    elif code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
        condition = "rainy"
        suggested_mood = "melancholic or dark"
    elif code in [71, 73, 75, 77]:
        condition = "snowy"
        suggested_mood = "romantic or uplifting"
    elif code in [95, 96, 99]:
        condition = "stormy"
        suggested_mood = "tense or dark"
    else:
        condition = "cloudy"
        suggested_mood = "thought-provoking"

    return {
        "city": city,
        "temperature_c": temp,
        "condition": condition,
        "suggested_mood": suggested_mood,
        "recommendation": (
            f"It is {temp}°C and {condition} in {city}. "
            f"Perfect weather for a {suggested_mood} film."
        )
    }


print("✓ Function 1 ready — get_movie_details (TMDB)")
print("✓ Function 2 ready — get_awards_data (OMDb)")
print("✓ Function 3 ready — get_weather_mood (Open-Meteo)")
print("\nTesting all 3 functions...")

# Quick test
# test1 = get_movie_details("Oldboy")
# print(f"\n TMDB test: {test1['title']} "
#       f"({test1['year']}) — dir. {test1['director']}")

# test2 = get_awards_data("Oldboy")
# print(f" OMDb test: {test2['title']} — {test2['awards'][:60]}")

# test3 = get_weather_mood("Dublin")
# print(f" Weather test: {test3['recommendation']}")

✓ Function 1 ready — get_movie_details (TMDB)
✓ Function 2 ready — get_awards_data (OMDb)
✓ Function 3 ready — get_weather_mood (Open-Meteo)

Testing all 3 functions...

 TMDB test: Oldboy (2003) — dir. Park Chan-wook
 OMDb test: Oldboy — 46 wins & 28 nominations total
 Weather test: It is 14.2°C and rainy in Dublin. Perfect weather for a melancholic or dark film.


In [ ]:
tools = [
    # ── Tool 1: File Search (RAG) ──────────────────────
    {
        "type": "file_search",
        "vector_store_ids": [VECTOR_STORE_ID]
    },

    # ── Tool 2: TMDB live movie details ────────────────
    {
        "type": "function",
        "name": "get_movie_details",
        "description": (
            "Fetch live details about a specific movie "
            "including director, cast, runtime, and rating. "
            "Use this when the user asks about a specific film."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "title": {
                    "type": "string",
                    "description": "The movie title to look up"
                }
            },
            "required": ["title"],
            "additionalProperties": False
        },
        "strict": True
    },

    # ── Tool 3: OMDb awards data ───────────────────────
    {
        "type": "function",
        "name": "get_awards_data",
        "description": (
            "Fetch awards, Metacritic score, and additional "
            "critical info about a film from OMDb. "
            "Use this when user asks about awards, "
            "critical reception, or wants to know if a "
            "film won any prizes."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "title": {
                    "type": "string",
                    "description": "The movie title"
                }
            },
            "required": ["title"],
            "additionalProperties": False
        },
        "strict": True
    },

    # ── Tool 4: Weather mood matcher ───────────────────
    {
        "type": "function",
        "name": "get_weather_mood",
        "description": (
            "Get the current weather in a city and suggest "
            "a matching film mood. Use this when the user "
            "asks what to watch today, or mentions their "
            "location, or asks for weather-based suggestions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": (
                        "City name e.g. Dublin, London, Paris. "
                        "If not mentioned ask the user."
                    )
                }
            },
            "required": ["city"],
            "additionalProperties": False
        },
        "strict": True
    }
]

print("✓ Tools updated with vector store ID")
print(f"✓ Vector Store ID attached: {VECTOR_STORE_ID}")
print(f"✓ Total tools: {len(tools)}")

✓ Tools updated with vector store ID
✓ Vector Store ID attached: vs_69f0d91457ac819185f331656b10b225
✓ Total tools: 4


In [ ]:
# ============================================
# SYSTEM PROMPT — CineMatch's personality
# ============================================
SYSTEM_PROMPT = """
You are CineMatch, an expert personal film analyst
and hidden gem specialist. You have deep knowledge
of world cinema, independent film, and international
cinema that most people have never discovered.

Your personality:
- Passionate and knowledgeable about cinema
- You recommend hidden gems over mainstream films
- Honest and specific — never vague
- You explain WHY a film matches someone's taste
- You reference directors, themes, and moods precisely

Your capabilities:
1. Search your curated database of 223 hidden gem
   films using file_search
2. Fetch live movie details via get_movie_details
3. Get awards and critical info via get_awards_data
4. Match films to current weather/mood via
   get_weather_mood

Your rules:
- ALWAYS search your film database first before
  recommending anything
- When a user mentions a city or asks what to watch
  today — ALWAYS call get_weather_mood first
- When a user asks about a specific film — ALWAYS
  call get_movie_details
- When a user asks about awards or critical reception
  — ALWAYS call get_awards_data
- Never recommend mainstream blockbusters unless
  specifically asked
- Always explain the hidden gem score when relevant
- Keep responses focused and specific — no waffle

Your opening message when conversation starts:
Greet the user warmly, introduce yourself as CineMatch,
and ask them two things:
1. What kind of films do they usually enjoy?
2. Where are they located today?
"""

print("✓ System prompt ready")
print(f"✓ Length: {len(SYSTEM_PROMPT)} characters")

✓ System prompt ready
✓ Length: 1440 characters


In [ ]:
import json

def run_function(name: str, arguments: str) -> str:
    """
    Execute whichever function the LLM requests
    and return the result as a string
    """
    args = json.loads(arguments)

    if name == "get_movie_details":
        result = get_movie_details(args["title"])

    elif name == "get_awards_data":
        result = get_awards_data(args["title"])

    elif name == "get_weather_mood":
        result = get_weather_mood(args["city"])

    else:
        result = {"error": f"Unknown function: {name}"}

    return json.dumps(result)


print("✓ Function dispatcher ready")

✓ Function dispatcher ready


In [ ]:
def run_cinematch(user_message: str,
                  conversation_history: list) -> str:
    """
    Main CineMatch agent loop.
    Handles RAG + function calls + conversation memory.
    """
    # Add user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Keep looping only for real function calls
    # file_search is handled automatically — no loop for it
    while True:
        response = client.responses.create(
            model="gpt-4.1",
            instructions=SYSTEM_PROMPT,
            input=conversation_history,
            tools=tools,
            tool_choice="auto",
            temperature=0.7,
            max_output_tokens=1000,
        )

        # Check if there are any real function calls
        # (NOT file_search — that's automatic)
        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        # If no function calls — LLM has final answer
        if not function_calls:
            final_response = response.output_text

            # Add to conversation history
            conversation_history.append({
                "role": "assistant",
                "content": final_response
            })

            return final_response

        # Execute each real function call
        for item in function_calls:
            print(f"  → Calling {item.name}...")

            fn_result = run_function(
                item.name, item.arguments
            )

            # Add function call to history
            conversation_history.append(item)

            # Add result to history
            conversation_history.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": fn_result
            })

✓ Agent loop fixed — file_search handled automatically


In [ ]:
# Test the agent with one message
print("=" * 50)
print("CINEMATCH — FILM INTELLIGENCE AGENT")
print("=" * 50)

# Start fresh conversation
conversation = []

# Send first message
test_response = run_cinematch(
    "Hi! I love psychological thrillers and "
    "dark drama films. I'm based in Dublin.",
    conversation
)

print("\nCineMatch:")
print(test_response)

CINEMATCH — FILM INTELLIGENCE AGENT
  → Calling get_weather_mood...

CineMatch:
Fantastic taste! With the rainy, melancholic Dublin weather and your love for psychological thrillers and dark drama, I’ve got some truly hidden gems that fit the mood and your interests perfectly:

---

### 1. **Forgotten** (2017, South Korea)
**Genre:** Thriller, Mystery, Crime  
**Hidden Gem Score:** 8.2/10  
**Why you’ll love it:** This is a tense, mind-bending psychological thriller that plays tricks with memory and identity. It’s got a rainy, paranoid atmosphere and plenty of twists—you’ll be guessing until the end. If you liked “Oldboy” or “Prisoners,” this is a must-watch, but far less known .

---

### 2. **Perfect Blue** (1998, Japan)
**Genre:** Animation, Thriller  
**Hidden Gem Score:** 7.7/10  
**Why you’ll love it:** Often described as one of the best psychological thrillers ever, this animated film by Satoshi Kon delves into obsession, fame, and the blurring of reality and hallucination. Its 

In [ ]:
def analyse_watch_history(csv_text: str) -> str:
    """
    Use code interpreter to analyse user's watch history.
    Takes CSV text as string, returns analysis + charts.
    """

    # Upload the watch history as a file to OpenAI
    import io
    csv_bytes = csv_text.encode('utf-8')

    watch_file = client.files.create(
        file=("watch_history.csv",
              io.BytesIO(csv_bytes),
              "text/csv"),
        purpose="assistants"
    )

    print(f"✓ Watch history uploaded: {watch_file.id}")

    # Create a code interpreter request
    analysis_response = client.responses.create(
        model="gpt-4.1",
        instructions="""
        You are a film taste analyst. Analyse the user's
        watch history CSV and produce:
        1. Genre distribution chart (bar chart)
        2. Rating distribution (histogram)
        3. Top 3 taste patterns/biases discovered
        4. Hidden gem score — how mainstream vs indie
           is their taste (0-10)
        5. One sentence summary of their film personality

        Be specific and insightful. Reference actual films
        from their history to support your findings.
        """,
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": (
                        "Analyse my complete watch history "
                        "and give me deep insights about "
                        "my film taste and patterns."
                    )
                },
                {
                    "type": "input_file",
                    "file_id": watch_file.id
                }
            ]
        }],
        tools=[{"type": "code_interpreter",
                "container": {"type": "auto"}}]
    )

    # Clean up uploaded file
    client.files.delete(watch_file.id)

    return analysis_response.output_text


print("✓ Code interpreter analyser ready")

✓ Code interpreter analyser ready


In [ ]:
# Sample watch history CSV for testing
# In real use the user uploads their own
sample_watch_history = """title,year,genre,my_rating,times_watched
Oldboy,2003,Thriller,9,2
City of God,2002,Crime Drama,8,1
Parasite,2019,Thriller,9,3
The Lives of Others,2006,Drama,8,1
Perfect Blue,1998,Animation Thriller,9,2
Memories of Murder,2003,Crime,8,1
Spirited Away,2001,Animation Fantasy,9,4
Burning,2018,Drama Mystery,7,1
A Taxi Driver,2017,Drama,8,1
The Handmaiden,2016,Thriller,9,2
Pan's Labyrinth,2006,Fantasy Drama,8,2
Capernaum,2018,Drama,8,1
Roma,2018,Drama,8,1
Portrait of a Lady on Fire,2019,Romance Drama,8,1
Shoplifters,2018,Drama,9,2
"""

print("Running watch history analysis...")
print("(This may take 30-60 seconds — code interpreter is working)\n")

analysis = analyse_watch_history(sample_watch_history)
print(analysis)

Running watch history analysis...
(This may take 30-60 seconds — code interpreter is working)

✓ Watch history uploaded: file-1iEjVSWjpdph7Z7CvtNAXR
There was a visualization error with the genre bar chart coloring. Let me quickly fix this and continue the analysis and visual insights for your film tastes.


In [ ]:
import gradio as gr

# Full conversation history stored here
conversation_history = []

def chat(user_message, history):
    """
    Gradio chat function.
    Connects the UI to the CineMatch agent loop.
    """
    global conversation_history

    # Run CineMatch agent
    response = run_cinematch(
        user_message,
        conversation_history
    )

    # Append to history in correct format
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": response})

    return history, ""


def reset_conversation():
    """Clear conversation history for fresh start"""
    global conversation_history
    conversation_history = []
    return [], ""


# Build the Gradio interface
with gr.Blocks(
    title="CineMatch",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎬 CineMatch — Personal Film Intelligence
    ### Your AI guide to hidden gems and world cinema
    *Powered by a curated database of 223 obscure films,
    live TMDB & OMDb data, and weather-based mood matching*
    """)

    chatbot = gr.Chatbot(
        label="CineMatch",
        height=500,
        type="messages"
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder=(
                "Ask me anything about films... "
                "e.g. 'I love dark thrillers, I'm in Dublin'"
            ),
            label="Your message",
            scale=4
        )
        send = gr.Button("Send 🎬", scale=1)

    with gr.Row():
        reset = gr.Button(
            "🔄 New Conversation",
            variant="secondary"
        )

    gr.Markdown("""
    **Try these:**
    - *"I love psychological thrillers, I'm in Dublin"*
    - *"Tell me about the film Oldboy"*
    - *"Did City of God win any awards?"*
    - *"Find me a hidden gem with score above 8"*
    """)

    # Connect buttons
    send.click(
        fn=chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    msg.submit(
        fn=chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    reset.click(
        fn=reset_conversation,
        outputs=[chatbot, msg]
    )

# Launch
print("Launching CineMatch...")
demo.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_5190/1772566072.py:34: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5190/1772566072.py:46: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Launching CineMatch...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://496c5445f1dc3cec0c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
